# 🏰 Holdfast — Full-Duplex-Bench Reproduction

This notebook runs the Holdfast voice agent project on Google Colab.

**Requirements:**
- GPU runtime (T4 is fine) — go to `Runtime > Change runtime type > T4 GPU`
- API keys: LiveKit Cloud + OpenAI (only needed for the full benchmark)

**Sections:**
1. Upload & Setup
2. Install Dependencies
3. Offline Tests (free, no API keys)
4. Full Benchmark (needs API keys)
5. Download Results

---
## 1️⃣ Upload the Project

**Option A:** Upload a zip of the `holdfast` folder (recommended if no GitHub repo).

**Option B:** Clone from GitHub (uncomment and edit the git clone line).

In [ ]:
import os

# === OPTION A: Upload a zip file ===
# Run this cell, then click the upload button that appears.
# Upload a zip named "holdfast.zip" containing the project.

from google.colab import files
uploaded = files.upload()  # Upload holdfast.zip here

!unzip -qo holdfast.zip -d /content/holdfast_project

# Auto-detect the project root (handles nested folders)
import glob
candidates = glob.glob('/content/holdfast_project/**/reproduce.sh', recursive=True)
if candidates:
    PROJECT_ROOT = os.path.dirname(candidates[0])
else:
    raise FileNotFoundError('Could not find reproduce.sh — check your zip structure')

print(f'\n✅ Project root: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)
!ls -la

In [ ]:
# === OPTION B: Clone from GitHub (uncomment and edit) ===
# !git clone https://github.com/YOUR_USERNAME/holdfast.git /content/holdfast
# %cd /content/holdfast
# PROJECT_ROOT = '/content/holdfast'

---
## 2️⃣ Preflight Checks & Install Dependencies

In [ ]:
# Check prerequisites
!echo "Python:" && python --version
!echo "\nFFmpeg:" && ffmpeg -version 2>&1 | head -1
!echo "\nGit:" && git --version
!echo "\nGPU:" && nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'No GPU detected — switch to GPU runtime!'

In [ ]:
# Install agent environment
!python -m venv .venv-agent
!.venv-agent/bin/pip install -q --upgrade pip
!.venv-agent/bin/pip install -q -r requirements-agent.txt
print('\n✅ Agent environment installed')

---
## 3️⃣ Offline Tests (No API Keys Needed)

These verify the control layer logic without making any API calls.

In [ ]:
# Run the 93 unit tests
!.venv-agent/bin/python -m pytest -q tests

In [ ]:
# Run the harness self-test (3 scripted scenarios)
!.venv-agent/bin/python scripts/harness_selftest.py

In [ ]:
# Run the Drive extension simulation
!.venv-agent/bin/python scripts/drive_sim.py

---
## 4️⃣ Full Benchmark (Needs API Keys)

⚠️ **This costs real money** — it uses the OpenAI Realtime API for the agent and gpt-4o for the judge.

You need:
- **LiveKit Cloud** — free project at https://cloud.livekit.io
- **OpenAI API key** — with access to the Realtime API

### Set your API keys
Use Colab Secrets (🔑 in sidebar) for security, or paste directly below.

In [ ]:
# === METHOD 1: Using Colab Secrets (recommended) ===
# Add these 4 secrets in the 🔑 sidebar: LIVEKIT_URL, LIVEKIT_API_KEY, LIVEKIT_API_SECRET, OPENAI_API_KEY
# Then uncomment the lines below:

# from google.colab import userdata
# LIVEKIT_URL = userdata.get('LIVEKIT_URL')
# LIVEKIT_API_KEY = userdata.get('LIVEKIT_API_KEY')
# LIVEKIT_API_SECRET = userdata.get('LIVEKIT_API_SECRET')
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# === METHOD 2: Paste directly (less secure) ===
LIVEKIT_URL = "wss://your-project.livekit.cloud"  # <-- edit this
LIVEKIT_API_KEY = ""                                # <-- edit this
LIVEKIT_API_SECRET = ""                             # <-- edit this
OPENAI_API_KEY = ""                                 # <-- edit this

# Write .env file
with open('.env', 'w') as f:
    f.write(f'LIVEKIT_URL={LIVEKIT_URL}\n')
    f.write(f'LIVEKIT_API_KEY={LIVEKIT_API_KEY}\n')
    f.write(f'LIVEKIT_API_SECRET={LIVEKIT_API_SECRET}\n')
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')

# Validate
missing = [k for k, v in {'LIVEKIT_URL': LIVEKIT_URL, 'LIVEKIT_API_KEY': LIVEKIT_API_KEY,
           'LIVEKIT_API_SECRET': LIVEKIT_API_SECRET, 'OPENAI_API_KEY': OPENAI_API_KEY}.items()
           if not v or v.startswith('wss://your')]
if missing:
    print(f'⚠️  Missing keys: {", ".join(missing)} — fill them in before running the benchmark!')
else:
    print('✅ All API keys configured')

In [ ]:
# Install the harness environment (includes NeMo ASR — takes 5-10 min)
!python -m venv .venv-harness
!.venv-harness/bin/pip install -q --upgrade pip
!.venv-harness/bin/pip install -q -r requirements-harness.txt
print('\n✅ Harness environment installed')

In [ ]:
# Run the FULL benchmark (1 run)
# For statistical significance, set RUNS=3 (takes 3x longer and costs 3x more)
!chmod +x reproduce.sh
!RUNS=1 ./reproduce.sh

---
## 5️⃣ View & Download Results

In [ ]:
# Show the summary
import glob
summaries = sorted(glob.glob('results/runs/*/summary.md'))
if summaries:
    latest = summaries[-1]
    print(f'📄 Latest results: {latest}\n')
    with open(latest) as f:
        print(f.read())
else:
    print('No results found yet — run the benchmark first (Section 4)')

In [ ]:
# Download all results as a zip
!zip -qr /content/holdfast_results.zip results/

from google.colab import files
files.download('/content/holdfast_results.zip')
print('\n✅ Results downloaded!')